# Character Recognition (B, 0, E) — End-to-End Pipeline

Run this notebook from the **project root** (`Assignment/`), or set `os.chdir` below.

Pipeline: **Part 1** dataset → **Part 2** train 64-3-3 → **Part 3** weight plots → **Part 5** architecture search (optional, longer).

In [ ]:
import os
import sys

# Project root (parent of notebooks/)
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)
else:
    sys.path.insert(0, os.getcwd())

print("Working directory:", os.getcwd())

## Part 1 — Generate dataset

In [ ]:
%matplotlib inline
from data_generation import generate_dataset, save_dataset, plot_templates, SAMPLES_PER_CLASS

os.makedirs("data", exist_ok=True)
os.makedirs("plots", exist_ok=True)

X, y, labels = generate_dataset(n_per_class=SAMPLES_PER_CLASS, seed=42)
save_dataset(X, y, labels, out_dir="data")
plot_templates(save_path="plots/templates.png");

## Part 2 — Train 64-3-3 network

In [ ]:
import numpy as np
from data_generation import load_dataset
from model import NeuralNetwork
from train import train, train_val_split, evaluate, plot_training_curves, plot_confusion_matrix

os.makedirs("models", exist_ok=True)

X, y, labels = load_dataset("data")
X_tr, y_tr, l_tr, X_v, y_v, l_v = train_val_split(X, y, labels, val_frac=0.2, seed=7)

net = NeuralNetwork(input_dim=64, hidden_dim=3, output_dim=3, seed=42)
history = train(net, X_tr, y_tr, l_tr, X_v, y_v, l_v, epochs=3000, batch_size=32,
                lr=0.001, patience=300, seed=0, verbose=True)
net.save("models/net_64_3_3")

t_loss, t_acc = evaluate(net, X_tr, y_tr, l_tr)
v_loss, v_acc = evaluate(net, X_v, y_v, l_v)
print(f"Train acc {t_acc*100:.2f}% | Val acc {v_acc*100:.2f}%")

plot_training_curves(history, title="64-3-3", save_path="plots/training_curves.png");
plot_confusion_matrix(net, np.vstack([X_tr, X_v]), np.concatenate([l_tr, l_v]),
                      save_path="plots/confusion_matrix.png");

## Part 3 — Weight visualisation

In [ ]:
import numpy as np
from visualize_weights import (
    plot_input_hidden_weights, plot_input_hidden_overlay,
    plot_hidden_output_weights, plot_biases, print_interpretation,
)
from model import NeuralNetwork

net = NeuralNetwork.load("models/net_64_3_3")
plot_input_hidden_weights(net, save_path="plots/weights_input_hidden.png");
plot_input_hidden_overlay(net, save_path="plots/weights_overlay.png");
plot_hidden_output_weights(net, save_path="plots/weights_hidden_output.png");
plot_biases(net, save_path="plots/biases.png");
print_interpretation(net)

## Part 5 — Architecture search (optional; several minutes)

Uncomment to run. Same as `python architecture_search.py`.

In [ ]:
# Long-running: trains all hidden sizes and sample-complexity sweeps.
# From project root in a terminal:
#   python architecture_search.py